# GPT mini tiếng Việt: từ checkpoint PyTorch tới GGUF chạy bằng llama.cpp (portfolio)

Bạn đưa GPT mini của module 05 đi hết con đường mà mọi model mở đi trước khi tới tay người dùng: xuất sang **GGUF**, quantize
bằng **llama.cpp** ở nhiều mức (`Q8_0`, `Q4_K_M`, `Q2_K`...), chạy bằng `llama-cpp-python` (hoặc Ollama), và so với các
phương pháp tự cài (RTN, GPTQ, AWQ, QAT) trên cùng validation: bpc, dung lượng, tốc độ sinh token. Kết quả là một bảng đánh đổi
có điều kiện đo rõ ràng và một tệp GGUF chạy được.

**Cách làm bài**
- Ba hàm được test chấm (cell dưới, đặc tả trong docstring). Phần còn lại tự thiết kế.
- `llama-cpp-python` nằm ở nhóm tuỳ chọn: `uv sync --group llama` (build từ source khoảng 4 phút, cần gcc/g++; cmake được kéo
  từ PyPI nếu máy chưa có). Không build được thì dùng Ollama (`ollama create ... -f Modelfile` với `FROM ./model.gguf`).
- Chấm: `uv run pytest tracks/ai-engineer/modules/10-quantization/exercises/advanced/01-gpt-mini-gguf-portfolio -q`

In [ ]:
# explore: giới hạn luồng, tìm gốc repo, kiểm Wikipedia tiếng Việt (tải một lần bằng env gốc), chỗ để checkpoint
import os

os.environ.setdefault("OMP_NUM_THREADS", "4")
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
WIKI = ROOT / "data/raw/wikipedia-vi"
if not (WIKI.is_dir() and any(WIKI.iterdir())):
    from data.download_dataset import ensure

    ensure("wikipedia-vi")
SHARD = WIKI / "20231101.vi/train-00002-of-00004.parquet"
CKPT = Path(os.environ.get("GPT_MINI_CKPT", ROOT / "data/raw/gpt-mini-vi/gpt_mini_vi.pt"))
FALLBACK = Path(os.environ.get("GPT_MINI_FALLBACK", ROOT / "data/raw/gpt-mini-vi/fallback_small.pt"))
print("checkpoint:", CKPT, "có" if CKPT.exists() else "chưa có")

In [ ]:
import json

import gguf
import numpy as np
import torch

## 1. Ba hàm được chấm

GGUF là một tệp nhị phân: header, các cặp khoá-giá trị (metadata: kiến trúc, siêu tham số, tokenizer), rồi các tensor có tên.
llama.cpp đã hỗ trợ kiến trúc `gpt2` (pre-norm, LayerNorm có bias, positional embedding học được, `qkv` gộp, GELU), đúng
kiến trúc GPT mini, nên chỉ cần đặt đúng tên tensor và metadata.

- `gguf_tensor_name(name)`: tên trong `state_dict` của GPT mini → tên tensor GGUF của kiến trúc `gpt2`:
  `tok_emb.weight` → `token_embd.weight`, `pos_emb.weight` → `position_embd.weight`, `ln_f.{weight,bias}` →
  `output_norm.{weight,bias}`, và với khối `i`: `ln1` → `blk.i.attn_norm`, `attn.qkv` → `blk.i.attn_qkv`, `attn.proj` →
  `blk.i.attn_output`, `ln2` → `blk.i.ffn_norm`, `mlp.fc` → `blk.i.ffn_up`, `mlp.proj` → `blk.i.ffn_down` (giữ hậu tố
  `.weight`/`.bias`). `lm_head.weight` trả `None` (dùng chung với `tok_emb`, llama.cpp tự dùng lại `token_embd` khi không có
  tensor `output`). Tên khác: `KeyError`.
- `export_gguf(model, tokenizer, path, dtype="f32")`: ghi GGUF kiến trúc `"gpt2"` bằng `gguf.GGUFWriter` với: tên
  `general.name`, `context_length` = `block_size`, `embedding_length` = `n_embd`, `block_count` = `n_layer`,
  `feed_forward_length` = `4 * n_embd`, `head_count` = `n_head`, `layer_norm_eps` = `1e-5`, `file_type` hợp `dtype`;
  tokenizer: model `"gpt2"`, pre-tokenizer `"gpt-2"`, danh sách token theo đúng thứ tự id, token type `NORMAL`, danh sách
  merge dạng `"a b"`, không tự thêm BOS. Mọi tensor (trừ `lm_head`) dưới tên `gguf_tensor_name`, dạng numpy cùng shape với
  PyTorch; `dtype="f16"` thì ma trận 2 chiều lưu float16, vector 1 chiều vẫn float32.
- `sequence_nll(logits, targets)`: `logits` numpy shape `(T, V)`, `targets` shape `(T,)`; trả tổng negative log-likelihood
  (nat) kiểu `float`, tính bằng log-softmax ổn định số trong float64. Dùng để tính bpc từ logits mà llama.cpp trả về, cùng định
  nghĩa với `evaluate_nll` của module 05.

In [ ]:
def gguf_tensor_name(name: str):
    # TODO
    raise NotImplementedError


def export_gguf(model, tokenizer, path, dtype: str = "f32") -> None:
    # TODO: gguf.GGUFWriter(path, "gpt2"), các hàm add_*, add_tensor, rồi write_header_to_file / write_kv_data_to_file /
    # write_tensors_to_file / close
    raise NotImplementedError


def sequence_nll(logits: np.ndarray, targets: np.ndarray) -> float:
    # TODO
    raise NotImplementedError

## 2. Gợi ý bố cục (không chấm)

1. **Nạp** checkpoint `gpt-mini-v1` của bạn (cell nạp giống bài Intermediate 01) và validation 100 bài của module 04/05.
2. **Xuất** f32 và f16; nạp lại bằng `llama_cpp.Llama(..., logits_all=True)`, kiểm tokenizer của llama.cpp cho cùng token id với
   tokenizer gốc trên vài câu tiếng Việt, và bpc của bản f32 chạy bằng llama.cpp khớp bpc PyTorch (cùng cửa sổ trượt, đưa thẳng
   token id vào `llm.eval`, đọc `llm.scores`).
3. **Quantize** bằng `llama_cpp.llama_model_quantize` (hoặc `ollama create --quantize`): ít nhất `Q8_0`, `Q4_0`, `Q4_K_M`,
   `Q2_K`. Đọc lại kiểu từng tensor bằng `gguf.GGUFReader`: tensor nào không ra đúng kiểu bạn yêu cầu, vì sao?
4. **Đo** trên cùng validation: bpc, dung lượng tệp, tốc độ sinh token (ghi rõ CPU/GPU, số luồng, máy có đang bận không);
   thêm các dòng tự cài (RTN, GPTQ, AWQ, QAT của bài Intermediate) ở cùng số bit; tuỳ chọn bitsandbytes trên GPU.
5. Viết `REPORT.md`.

In [ ]:
# explore: TODO: pipeline của bạn